In [5]:
import pandas as pd
from windpowerlib import ModelChain, WindTurbine, create_power_curve, wind_speed
from windpowerlib import data as wt
import numpy as np

# Define the folder path where your wind scenario files are located
folder_path = 'path_to_your_folder'  # Replace with the actual folder path

# Define turbine specifications
enercon_e53 = {
    'turbine_type': 'E-53/800',
    'hub_height': 100  # in m
}

# Initialize WindTurbine object
e53 = WindTurbine(**enercon_e53)
rated_power = 800000

# TO REPLACE BY A SMALLER WIND TURBINE!

# ModelChain specifications
modelchain_data = {
    'wind_speed_model': 'logarithmic',
    'density_model': 'ideal_gas',
    'temperature_model': 'linear_gradient',
    'power_output_model': 'power_coefficient_curve',
    'density_correction': True,
    'obstacle_height': 0,
    'hellman_exp': None
}

N = 20

# Loop through wind scenario files and process each one
for scenario_number in range(1, N):  # Replace N with the total number of scenarios
    # Read wind scenario file
    file_name = f'wind_scenario_{scenario_number}.csv'
    df = pd.read_csv(file_name, header=None)
    df.columns = ['Wind Speed (m/s)']

    # Generate timestamps for each hour in a year
    start_date = '2005-01-01'
    timestamps = pd.date_range(start=start_date, periods=len(df), freq='H')
    df['Datetime'] = timestamps

    # Add a new column 'Air Density (kg/m^3)' with all values set to 1.225
    df['Air Density (kg/m^3)'] = 1.225

    # Convert wind speed to hub height
    windspeed = df['Wind Speed (m/s)']
    windspeed_hub = wind_speed.hellman(windspeed, 10, 100)

    # Create ModelChain and calculate power output
    power_output = ModelChain(e53, **modelchain_data).calculate_power_output(windspeed_hub, df['Air Density (kg/m^3)'])

    # Create a DataFrame for power timeseries
    power_timeseries = pd.DataFrame({
        'P': power_output / rated_power,  # Power values
        'Datetime': timestamps  # Date and hour
    })

    # Save power timeseries as CSV file
    output_file_name = f'windpower_scenario_{scenario_number}.csv'
    power_timeseries.to_csv(output_file_name, index=False)
